In [ ]:
!pip install -q transformers numpy accelerate datasets pandas evaluate

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from datasets import Dataset
from evaluate import load
from scipy.stats import pearsonr
from google.colab import drive


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive/ColabNotebooks'))


In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/train_data.csv')
test_df = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/test_data.csv')

In [ ]:
MODEL_NAME = "readerbench/RoGPT2-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

model.eval()


In [ ]:
def extract_features(df):
  df = df.copy()
  df["clean_word"] = (
    df["word"].fillna("")
      .str.replace("ţ", "ț")
      .str.replace("ş", "ș")
      .str.replace("Ţ", "Ț")
      .str.replace("Ş", "Ș")
  )

  df["group_key"] = df["word_id"].astype(str).str.rsplit("_", n=1).str[0]
  df["word_idx"] = df["word_id"].astype(str).str.rsplit("_", n=1).str[1].astype(int)

  unique_words = (
    df[["group_key", "word_idx", "clean_word"]]
    .drop_duplicates()
    .sort_values(by=["group_key", "word_idx"])
  )

  surprisal_map = {}

  for group_key, group in unique_words.groupby("group_key"):
    words = group["clean_word"].tolist()
    indices = group["word_idx"].tolist()

    text, spans = "", []
    for w in words:
      spans.append((len(text), len(text) + len(w)))
      text += w + " "

    text = text.strip()

    enc = tokenizer(
        text,
        return_tensors = "pt",
        return_offsets_mapping = True,
        truncation = True,
        max_length = 1024
    )

    input_ids = enc["input_ids"].to(device)
    offsets = enc["offset_mapping"][0].tolist()

    with torch.no_grad():
      logits = model(input_ids = input_ids).logits[0]

      log_probs = torch.log_softmax(logits, dim = -1)

    # surprisal of token = -log P(token | previous token)

    tok_surp = [0.0] + [-log_probs[t - 1, input_ids[0, t]].item() for t in range(1, len(offsets))]

    for w_idx, (w_start, w_end) in zip(indices, spans):
      surprisal_map[(group_key, w_idx)] = sum(
          tok_surp[t] for t, (t_start, t_end) in enumerate(offsets)
          if t_end > w_start and t_start < w_end
      )
  keys = list(zip(df["group_key"], df["word_idx"]))
  df["surprisal"] = [surprisal_map.get(k, 0.0) for k in keys]
  df["word_len"] = df["clean_word"].str.len()
  df["surp_per_char"] = df["surprisal"] / (df["word_len"] + 1)
  df["is_punct"] = df["clean_word"].str.contains(r"[.,;:!?–-]", regex=True).astype(int)
  df["is_upper"] = df["clean_word"].str[0].str.isupper().fillna(False).astype(int)
  df["has_digit_or_url"] = df["clean_word"].str.contains(r"[0-9/]", regex=True).astype(int)

  df = df.sort_values(["participant_id", "group_key", "word_idx"]).reset_index(drop=True)
  df["prev_word_len"] = df.groupby(["participant_id", "group_key"])["word_len"].shift(1).fillna(0)
  df["prev_surprisal"] = df.groupby(["participant_id", "group_key"])["surprisal"].shift(1).fillna(0)
  df["next_word_len"] = df.groupby(["participant_id", "group_key"])["word_len"].shift(-1).fillna(0)
  return df


In [ ]:
train_feat = extract_features(train_df)
test_feat = extract_features(test_df)

In [ ]:
features = [
    "surprisal", "word_len", "surp_per_char", "is_punct", "is_upper",
    "has_digit_or_url", "word_idx", "prev_word_len", "prev_surprisal", "next_word_len"
]

def eval_metric(y_true, preds):
    r2 = max(0.0, r2_score(y_true, preds, force_finite=True))
    pears = abs(pearsonr(y_true, preds)[0])
    return 100.0 * (pears + r2) / 2.0

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(train_df))

In [ ]:
print(train_feat.columns.tolist())

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

for tr, val in gkf.split(train_feat, train_feat["answer"], groups=train_feat["participant_id"]):
    reg = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=6, random_state=42)
    reg.fit(train_feat.iloc[tr][features], train_feat.iloc[tr]["answer"])
    oof[val] = reg.predict(train_feat.iloc[val][features])

oof = np.clip(oof, 0, None)

In [ ]:
print(f"Local CV Score (must be > 36 to beat baseline): {eval_metric(train_feat['answer'], oof):.2f} / 100")

In [ ]:
final_model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=6, random_state=42)
final_model.fit(train_feat[features], train_feat["answer"])

In [ ]:
test_feat["pred"] = np.clip(final_model.predict(test_feat[features]), 0, None)
test_out = test_df[["datapointID"]].merge(test_feat[["datapointID", "pred"]], on="datapointID", how="left")

In [ ]:
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_out["datapointID"],
    "prediction": np.round(test_out["pred"], 10)
})

OUTPUT_DIR = "/content/drive/MyDrive/ColabNotebooks/submission_rogpt2_calibrated.csv"
submission.to_csv(OUTPUT_DIR, index = False)
print(submission.head(10))